# Week 2: Steps 2, 3 & 4 — Decoding, DPO Preference Alignment & Architecture Comparison

**Course:** Natural Language Processing (NLP) — Midterm Project  
**Project:** *Teaching DistilGPT2 Cebuano: Fine-Tuning, Decoding & Alignment (No GPU Required)*  
**Repository:** [github.com/Pycnosis/cebuano-distilgpt2](https://github.com/Pycnosis/cebuano-distilgpt2)  
**Team Members:**
- **Nico** ([@Pycnosis](https://github.com/Pycnosis)) — Step 2: Decoding Benchmarks & KV-Cache Latency
- **Lej** ([@lejxz](https://github.com/lejxz)) — Step 3: DPO Dataset Curation & Preference Alignment
- **Joshua** ([@JoshyEyeBa11](https://github.com/JoshyEyeBa11)) — Step 4: Architecture Comparison & Final Integration Lead

---

### Executive Summary & Week 2 Scope
In Week 1, we adapted `distilgpt2` to Cebuano via LoRA ($r=32$, $\alpha=64$), reducing validation perplexity from **211.56** to **34.88** (-83.5%). In Week 2, we evaluate three complementary dimensions:
1. **Step 2 (Nico):** Decoding algorithms (Greedy, Top-$k$, Nucleus Top-$p$, Beam Search), KV-cache inference speedup ($O(T)$ vs. $O(T^2)$), and MoE memory trade-offs.
2. **Step 3 (Lej):** Direct Preference Optimization (DPO) on 38 original, native-speaker-verified Cebuano triplets, aligning the model without reward-model overhead.
3. **Step 4 (Joshua):** Architectural comparative analysis of learned positional embeddings (`wpe`) vs. modern RoPE and ALiBi, and causal decoder vs. bidirectional encoder modeling.

---
## Step 2 — Decoding Strategies & KV-Cache Latency Benchmark (Nico)

### 2.1 The Four Decoding Regimes
We compare four distinct autoregressive generation strategies across the canonical test prompts `P1`–`P5`:
1. **Greedy Decoding** (`do_sample=False`): Deterministic argmax selection; susceptible to repetitive n-gram basins (`sa kapulisan sa kapulisan...`).
2. **Top-k Sampling** (`top_k=50, temperature=0.8`): Truncates probability tail to $k=50$ candidates, introducing lexical variety.
3. **Nucleus (Top-p) Sampling** (`top_p=0.9, temperature=0.8`): Dynamically samples from the smallest set of tokens covering cumulative probability $p=0.9$.
4. **Beam Search** (`num_beams=4, no_repeat_ngram_size=2`): Joint sequence optimization with strict 2-gram repeat prevention.

In [ ]:
# Step 2 Decoding & KV-Cache Inspection
from pathlib import Path

results_dir = Path('../results')
print('Available Results Files in Repository:')
for p in sorted(results_dir.glob('step*')):
    print(f' - {p.name}')

### 2.2 KV-Cache Complexity & MoE Inference Note
- **KV-Cache Complexity:** Without caching, computing autoregressive self-attention scales quadratically as $\mathcal{O}(T^2)$ FLOPs across $T$ generated tokens. Caching Key and Value projections preserves past activations, achieving linear $\mathcal{O}(T)$ total compute.
- **Mixture-of-Experts (MoE) Note:** Sparse routing activates only top-$k$ experts per token, lowering active compute latency, yet requires **all** expert weights to fit in memory (VRAM) simultaneously.

---
## Step 3 — Direct Preference Optimization (DPO) Alignment (Lej)

### 3.1 Preference Dataset (`data/cebuano_dpo_preferences.jsonl`)
Curated by `@lejxz`, the dataset contains **38 original Cebuano triplets** (`32` train, `6` test) targeting **clean, idiomatic Cebuano continuation without English code-switching, repetitive loops, or non-word gibberish**.

In [ ]:
import json
from pathlib import Path

dpo_file = Path('../data/cebuano_dpo_preferences.jsonl')
if dpo_file.exists():
    with open(dpo_file, 'r', encoding='utf-8') as f:
        triplets = [json.loads(line) for line in f if line.strip()]
    print(f'Total DPO Triplets Loaded: {len(triplets)}')
    print('\nSample Triplet (First Pair):')
    print('Prompt:  ', triplets[0]['prompt'])
    print('Chosen:  ', triplets[0]['chosen'])
    print('Rejected:', triplets[0]['rejected'])

### 3.2 DPO Metrics & Behavioral Shift (`results/step3_dpo_metrics.json`)
DPO optimizes the analytical policy ratio directly without requiring a separate reward model or online PPO rollouts.

In [ ]:
metrics_file = Path('../results/step3_dpo_metrics.json')
if metrics_file.exists():
    with open(metrics_file, 'r', encoding='utf-8') as f:
        metrics = json.load(f)
    summary = metrics.get('summary', {})
    print('Step 3 DPO Quantitative Summary:')
    for k, v in summary.items():
        print(f' - {k}: {v}')

---
## Step 4 — Architecture Comparison (Joshua)

Full writeup available at [`report/step4_architecture_comparison.md`](../report/step4_architecture_comparison.md).

### 4.1 Positional Encoding Mechanisms
| Mechanism | Where Injected | Learned Params | Relative Distance? | Max Context Ceiling | Effective Cebuano Words |
| :--- | :--- | :---: | :---: | :---: | :---: |
| **DistilGPT2 (`wpe`)** | Input token embeddings | $786{,}432$ ($1024 \times 768$) | ✗ Absolute index | $1{,}024$ tokens | **$\approx 241$ words** |
| **RoPE (Su et al., 2024)** | Attention query/keys | **0** | ✓ Rotation angle $(m - n)$ | Flexible ($32\text{k}+$) | Thousands |
| **ALiBi (Press et al., 2022)**| Pre-softmax logits | **0** | ✓ Linear penalty $|i - j|$ | Strong zero-shot | Thousands |

> **Key Takeaway:** Because Cebuano exhibits high subword fragmentation (**4.25 tokens/word** vs. 2.08 in English), `distilgpt2`'s learned absolute embedding bottleneck constrains the effective narrative context to roughly 240 words before memory saturation.

### 4.2 Causal Decoder-Only vs. Bidirectional Encoder-Only
| Task Domain | Optimal Model Family | Self-Attention Mask | Rationale |
| :--- | :--- | :--- | :--- |
| **Autoregressive Continuation (Steps 1–3)** | **Decoder-Only (`distilgpt2`)** | Lower-triangular causal ($j \le i$) | Sequential left-to-right generation; enables KV-caching. |
| **CebuaNER Named Entity Tagging** | **Encoder-Only (`mBERT` / `XLM-R`)** | Bidirectional ($j \in [0, T-1]$) | Resolving entity boundaries requires right-hand syntactic context. |
| **Readability Classification** | **Encoder-Only** | Bidirectional | Pools full-passage contextual representation via `[CLS]`. |

---
## Consolidated Week 2 Synthesis & Deliverables

| Step | Primary Owner | Deliverable Files | Key Metric / Result |
| :--- | :--- | :--- | :--- |
| **Step 2** | Nico (`@Pycnosis`) | `scripts/step2_*.py`, `results/step2_*.md` | Decoding benchmarks & KV-cache speedup |
| **Step 3** | Lej (`@lejxz`) | `data/cebuano_dpo_preferences.jsonl`, `checkpoints/distilgpt2-cebuano-dpo/` | Test accuracy: `0.0% -> 33.3%`, $\Delta$ margin: `+8.35` |
| **Step 4** | Joshua (`@JoshyEyeBa11`) | `report/step4_architecture_comparison.md`, `report/final_report.md` | Positional embedding & encoder vs. decoder analysis |